In [4]:
import pandas as pd

file_path = '/home/abhajkhan/Code/mca-tkmce/bipolar-disorder-severity-assessment-system/notebooks/Bipolar_preprocessed.csv'
df = pd.read_csv(file_path)

display(df.head())

,age,age_onset,total_number_of_episodes,total_duration_of_illness_in_months,total_duration_of_all_episodes_in_months,total_duration_of_hypomanic_episodes_in_months,total_number_of_manic_episodes,total_number_of_hypomanic_episodes,total_number_of_mixed_episodes,total_number_of_depressive_episodes,...,antipsychotic_Olanzapine,antipsychotic_Risperidone,antipsychotic_Unknown,education_standard_Illiterate,education_standard_Other/Unknown,education_standard_Postgraduate,education_standard_Primary/Secondary,education_standard_Secondary/Pre-University,education_standard_Undergraduate,education_standard_Unknown
0,0.532967,0.293343,0.010967,0.692430,0.041212,0.139536,0.374633,0.209127,0.054453,0.003024,...,False,False,False,False,False,False,True,False,False,False
1,0.767857,0.644235,0.005982,0.598200,0.012910,0.031978,0.280829,0.033378,0.054453,0.003024,...,False,False,True,False,True,False,False,False,False,False
2,0.506868,0.346951,0.006979,0.583025,0.006951,0.031978,0.343365,0.033378,0.054453,0.002022,...,False,False,False,False,False,False,False,False,True,False
3,0.644231,0.507777,0.005982,0.584966,0.006951,0.031978,0.280829,0.033378,0.054453,0.003024,...,False,False,False,True,False,False,False,False,False,False
4,0.520604,0.361572,0.010967,0.585848,0.015889,0.031978,0.437170,0.033378,0.054453,0.003024,...,False,False,False,False,True,False,False,False,False,False


In [5]:
from sklearn.cluster import OPTICS
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
import numpy as np

# Assuming 'df' contains the preprocessed data ready for clustering.
# First, let's select only numerical columns for clustering.
numerical_df = df.select_dtypes(include=np.number)

# Handle potential NaN values by filling with the mean, as OPTICS doesn't handle them.
# Although the prompt says 'no need of preprocessing', NaN handling is crucial for clustering algorithms.
# If there are no NaNs, this step will have no effect.
numerical_df = numerical_df.fillna(numerical_df.mean())

# Initialize and fit the OPTICS model
# Adjusted parameters to reduce the number of clusters
# Increasing min_samples and min_cluster_size to 5, and xi to 0.05 (less sensitive)
optics_model = OPTICS(min_samples=5, xi=0.05, min_cluster_size=5, n_jobs=-1)
# n_jobs=-1 uses all available CPU cores

optics_model.fit(numerical_df)

# Extract cluster labels
# Noise points are labeled as -1
labels = optics_model.labels_

# Number of clusters in labels, ignoring noise if present.
n_clusters_ = len(set(labels)) - (1 if -1 in labels else 0)

print(f'Estimated number of clusters: {n_clusters_}')

Estimated number of clusters: 4


/home/abhajkhan/.local/lib/python3.12/site-packages/sklearn/cluster/_optics.py:1086: RuntimeWarning: divide by zero encountered in divide
  ratio = reachability_plot[:-1] / reachability_plot[1:]


In [6]:
# Evaluate the clusters only for data points that are part of a cluster (excluding noise - label -1)

# Filter out noise points
clustered_data = numerical_df[labels != -1]
clustered_labels = labels[labels != -1]

if n_clusters_ > 1:
    # Silhouette Score
    silhouette = silhouette_score(clustered_data, clustered_labels)
    print(f'Silhouette Score: {silhouette:.3f}')

    # Davies-Bouldin Index
    davies_bouldin = davies_bouldin_score(clustered_data, clustered_labels)
    print(f'Davies-Bouldin Index: {davies_bouldin:.3f}')

    # Calinski-Harabasz Index
    calinski_harabasz = calinski_harabasz_score(clustered_data, clustered_labels)
    print(f'Calinski-Harabasz Index: {calinski_harabasz:.3f}')
elif n_clusters_ == 1:
    print("Only one cluster found (excluding noise), cannot compute Silhouette, Davies-Bouldin, or Calinski-Harabasz scores.")
else:
    print("No clusters found (only noise points).")


Silhouette Score: 0.651
Davies-Bouldin Index: 0.599
Calinski-Harabasz Index: 50.514


In [7]:
# Count data points in each cluster
cluster_counts = pd.Series(labels).value_counts().sort_index()
print("Count of data points in each cluster:")
print(cluster_counts)

# Rename the noise cluster for clarity if it exists
if -1 in cluster_counts.index:
    cluster_counts = cluster_counts.rename(index={-1: 'Noise_Points'})
    print("\nUpdated cluster counts (Noise_Points are samples not assigned to any cluster):")
    print(cluster_counts)


Count of data points in each cluster:
-1    306
 0      8
 1     12
 2      8
 3      8
Name: count, dtype: int64

Updated cluster counts (Noise_Points are samples not assigned to any cluster):
Noise_Points    306
0                 8
1                12
2                 8
3                 8
Name: count, dtype: int64


### Differentiating Features
To find differentiating features, we can look at the mean values of each feature for each cluster. This will highlight which features have significantly different average values across clusters.

In [8]:
# Add cluster labels to the numerical DataFrame
numerical_df_with_clusters = numerical_df.copy()
numerical_df_with_clusters['cluster'] = labels

# Exclude noise points (-1) from this analysis if desired, or include them as a separate 'cluster'
# For now, let's include them to see how they differ as well.

# Calculate the mean of each feature for each cluster
cluster_means = numerical_df_with_clusters.groupby('cluster').mean()

print("Mean values of features for each cluster (including noise as cluster -1):\n")
display(cluster_means)

print("\nTo identify differentiating features, look for columns where the mean values vary significantly across different clusters.")


Mean values of features for each cluster (including noise as cluster -1):



,age,age_onset,total_number_of_episodes,total_duration_of_illness_in_months,total_duration_of_all_episodes_in_months,total_duration_of_hypomanic_episodes_in_months,total_number_of_manic_episodes,total_number_of_hypomanic_episodes,total_number_of_mixed_episodes,total_number_of_depressive_episodes,...,rate_prominence_of_mixed_episodes,mania_vs_dep_first5yr,mania_vs_dep_recent,insightinterepisode,occupational_func_5yr,number_of_ip_care_recieved,a_criterion_score,criteria_b_score,total_score,rate_response_to_lithium_treatment
cluster,,,,,,,,,,,,,,,,,,,,,
-1,0.469612,0.395679,0.014459,0.462909,0.020870,0.059050,0.295757,0.089516,0.117061,0.010026,...,0.282276,0.720651,0.703364,0.275808,0.262124,0.013102,0.864531,0.497770,0.743433,0.840126
0,0.445063,0.365539,0.005234,0.463876,0.009434,0.037369,0.304280,0.066331,0.054453,0.001145,...,0.063139,1.000000,1.000000,0.269062,0.223280,0.005350,0.938029,0.449178,0.838107,0.893658
1,0.461834,0.390848,0.013976,0.457218,0.020225,0.057961,0.294688,0.087817,0.113946,0.009615,...,0.272670,0.734667,0.718863,0.293663,0.257835,0.012675,0.869247,0.496555,0.747792,0.844695
2,0.274382,0.184289,0.015467,0.439821,0.020746,0.055286,0.286754,0.087755,0.113700,0.011370,...,0.306087,0.789583,0.764580,0.295461,0.245633,0.012690,0.864367,0.492848,0.747755,0.845089
3,0.306490,0.369491,0.003240,0.244574,0.005958,0.038701,0.249560,0.044362,0.054453,0.001145,...,0.092416,0.968750,1.000000,1.000000,0.136498,0.003633,0.984131,0.499976,0.824244,0.970202



To identify differentiating features, look for columns where the mean values vary significantly across different clusters.
